# AgriLink — Next-Basket Recommendation (NBR)

# AgriLink Customer Recommendation
This notebook is a standalone baseline. It runs with generated example data by default.
To use AgriLink data, set `USE_CSV = True` and update `CSV_PATH`. Expected columns are documented below.
Do not report results from synthetic data as real AgriLink performance.

**Goal:** predict products in a customer's next order from their earlier orders.
**Required real-data columns:** `CustomerID`, `OrderID`, `Product`, `Quantity`, `OrderDate`.
Each `OrderID` should represent one basket/order. `OrderDate` must reflect order chronology.
This notebook implements a transparent frequency-based baseline and evaluates it with a per-customer chronological split.

In [ ]:
import pandas as pd
import numpy as np
from collections import Counter
from sklearn.metrics import precision_score

USE_CSV = False
CSV_PATH = "/content/agrilink_orders.csv"
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

products = ["Apple","Banana","Beans","Brinjal","Cabbage","Carrot","Coriander",
            "Cucumber","Green Chilli","Onion","Orange","Papaya","Potato","Rice","Tomato"]
rows = []
for customer in range(1, 101):
    start = pd.Timestamp("2025-01-01")
    preferred = rng.choice(products, size=5, replace=False).tolist()
    for order_num in range(1, int(rng.integers(4, 9))):
        date = start + pd.Timedelta(days=order_num * int(rng.integers(4, 15)))
        basket = rng.choice(preferred, size=int(rng.integers(2, 5)), replace=False)
        for product in basket:
            rows.append({
                "CustomerID": customer,
                "OrderID": f"C{customer}_O{order_num}",
                "Product": product,
                "Quantity": int(rng.integers(1, 4)),
                "OrderDate": date
            })
df = pd.DataFrame(rows)

if USE_CSV:
    df = pd.read_csv(CSV_PATH)
    required = {"CustomerID","OrderID","Product","Quantity","OrderDate"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"CSV is missing required columns: {missing}")

df["OrderDate"] = pd.to_datetime(df["OrderDate"])
df = df.dropna(subset=["CustomerID","OrderID","Product","OrderDate"]).copy()
df["Product"] = df["Product"].astype(str)
df.head()

## 1. Build chronological baskets
We sort each customer's orders by date. Each order is represented as a set of products, so repeated lines for the same product within one order do not duplicate the target item.

In [ ]:
order_table = (
    df.groupby(["CustomerID","OrderID"], as_index=False)
      .agg(OrderDate=("OrderDate","min"),
           Basket=("Product", lambda x: sorted(set(x))))
      .sort_values(["CustomerID","OrderDate","OrderID"])
)
order_table.head()

## 2. Create train/test examples
For each customer with at least two orders, use all but their final order as history and hold out the final order as the target next basket. This is a simple leave-last-basket-out evaluation.

In [ ]:
examples = []
for customer_id, group in order_table.groupby("CustomerID"):
    group = group.sort_values(["OrderDate","OrderID"])
    if len(group) < 2:
        continue
    history = group.iloc[:-1]
    target = set(group.iloc[-1]["Basket"])
    history_baskets = history["Basket"].tolist()
    examples.append((customer_id, history_baskets, target))

len(examples)

## 3. Baseline recommender
Rank products by frequency across the customer's historical baskets. If history is too short, fill remaining slots with global training popularity. This is a baseline, not a deep-learning model.

In [ ]:
global_counts = Counter(
    product
    for _, history_baskets, _ in examples
    for basket in history_baskets
    for product in basket
)

def recommend_next_basket(history_baskets, top_k=5, fallback_counts=None):
    personal_counts = Counter(p for basket in history_baskets for p in basket)
    ranked = [p for p, _ in personal_counts.most_common()]
    fallback = fallback_counts or Counter()
    for p, _ in fallback.most_common():
        if p not in ranked:
            ranked.append(p)
    return ranked[:top_k]

for customer_id, history, target in examples[:3]:
    print(customer_id, "history:", history, "target:", sorted(target),
          "recommendations:", recommend_next_basket(history, 5, global_counts))

## 4. Evaluate with ranking metrics
Recall@K measures the fraction of products in the held-out basket retrieved in the top K. NDCG@K rewards relevant products appearing higher in the ranked list. This evaluation uses only earlier baskets to make each prediction.

In [ ]:
def recall_at_k(recs, target, k):
    recs_k = recs[:k]
    return len(set(recs_k) & set(target)) / max(len(target), 1)

def ndcg_at_k(recs, target, k):
    rel = [1 if p in target else 0 for p in recs[:k]]
    dcg = sum(r / np.log2(i + 2) for i, r in enumerate(rel))
    ideal_hits = min(len(target), k)
    idcg = sum(1 / np.log2(i + 2) for i in range(ideal_hits))
    return dcg / idcg if idcg else 0.0

results = []
for customer_id, history, target in examples:
    recs = recommend_next_basket(history, top_k=10, fallback_counts=global_counts)
    results.append({
        "CustomerID": customer_id,
        "Recall@5": recall_at_k(recs, target, 5),
        "Recall@10": recall_at_k(recs, target, 10),
        "NDCG@10": ndcg_at_k(recs, target, 10)
    })
metrics = pd.DataFrame(results)
metrics.drop(columns="CustomerID").mean().to_frame("Mean score")

## 5. Inspect a customer's prediction
Use a customer ID present in the dataset. The target basket is shown only for offline evaluation; a deployed recommender would not have access to it at prediction time.

In [ ]:
example_customer = examples[0][0]
customer_example = next(x for x in examples if x[0] == example_customer)
_, history, target = customer_example
print("Customer:", example_customer)
print("Past baskets:", history)
print("Predicted next basket:", recommend_next_basket(history, 5, global_counts))
print("Held-out actual basket (evaluation only):", sorted(target))

## Limitations and next steps
- This is a frequency baseline for next-basket prediction, not ReCANet, LightGCN, or a Transformer model.
- The example uses a leave-last-basket-out split; for robust research, consider multiple rolling temporal cutoffs.
- Avoid leakage: product popularity for evaluation should be computed only from training baskets. This notebook's global fallback is intentionally simple; for a strict experiment, build a separate training-only global count per split.
- Report results separately for synthetic data and real/public datasets.